# Phase 3 — Was my earlier "nearest" right?

PuneStay measured *straight-line* distance. This notebook measures *walking* distance on the OSM street network
and asks the planner's question: **which residents of Prabhag 11 cannot reach a PG or hostel on foot within 15 minutes?**

Steps (each writes its result back to PostGIS):

1. load the walking graph saved by Phase 1 and project it to UTM 43N (metres);
2. **isochrones** at the walking threshold around every service (network), and straight-line **buffers** of the same radius;
3. **H3 cells** at resolution 8 and 9 over the ward; for each cell centre the network distance to the nearest service;
4. **population** per cell from the WorldPop 2020 constrained 100 m raster;
5. Q6: 30 random points, straight line vs network; Q7: coverage by unit (ward / H3-8 / H3-9, the MAUP); Q8: 3.6 km/h re-run.

Assumptions (stated in ANSWERS.md): walking speed **4.8 km/h = 80 m/min**, threshold **15 min = 1,200 m** (urban ward);
elderly scenario **3.6 km/h = 60 m/min → 900 m**. Services both inside the ward and in the 2 km margin count as
destinations, because a hostel just across the ward boundary is still reachable.

In [ ]:
import json, math, sys, time, random
from pathlib import Path

import geopandas as gpd
import pandas as pd
import numpy as np
import networkx as nx
import osmnx as ox
import h3
import rasterio
import shapely
from shapely.geometry import Point, Polygon, MultiPoint, box
from shapely.ops import unary_union
from sqlalchemy import text
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "analysis" else Path.cwd()
sys.path.insert(0, str(ROOT / "analysis"))
from dbconf import engine_from_env
engine = engine_from_env()

UTM = 32643
SPEED_KMH = 4.8                       # brief: healthy adult
MINUTES = 15                          # urban threshold
THRESH_M = SPEED_KMH * 1000 / 60 * MINUTES          # 1200 m
SLOW_KMH = 3.6                        # elderly walker, Q8
THRESH_SLOW_M = SLOW_KMH * 1000 / 60 * MINUTES      # 900 m
SNAP_MAX_M = 150                      # a cell centre farther than this from any walkable street counts as "no access"
RNG_SEED = 20261119
print(f"threshold {THRESH_M:.0f} m at {SPEED_KMH} km/h, slow threshold {THRESH_SLOW_M:.0f} m at {SLOW_KMH} km/h")
print("h3", h3.__version__, "| osmnx", ox.__version__, "| networkx", nx.__version__)

## 1. Graph, services and ward from Phase 1

In [ ]:
G = ox.load_graphml(ROOT / "data" / "raw" / "walk_extent.graphml")
Gp = ox.project_graph(G, to_crs=f"EPSG:{UTM}")          # edge 'length' is already metres; node x,y now metres too
Gu = ox.convert.to_undirected(Gp)                        # pedestrians ignore one-way rules
print(f"walk graph: {Gu.number_of_nodes():,} nodes, {Gu.number_of_edges():,} edges")

with engine.connect() as con:
    services = gpd.read_postgis(text("SELECT service_id, name, service_type, inside_area, geom FROM services"), con, geom_col="geom", crs=4326)
    area = gpd.read_postgis(text("SELECT area_id, name, geom FROM area"), con, geom_col="geom", crs=4326)
services_u = services.to_crs(UTM)
area_u = area.to_crs(UTM)
ward = area_u.geometry.iloc[0]
print(f"services: {len(services)} ({int(services.inside_area.sum())} inside the ward)")

# snap every service to its nearest graph node; keep the snap distance (a hostel 300 m from any street is suspicious)
sx, sy = services_u.geometry.x.values, services_u.geometry.y.values
services_u["node"], services_u["snap_m"] = ox.distance.nearest_nodes(Gu, sx, sy, return_dist=True)
print("service snap distance to street (m): median %.0f, max %.0f" % (services_u.snap_m.median(), services_u.snap_m.max()))

## 2. Isochrones (network) and buffers (straight line) around every service

An isochrone here = every street segment reachable within 1,200 m of walking from the service's nearest node
(`networkx.ego_graph` with `distance="length"`), **buffered by 30 m** and dissolved. Buffering the reached *edges*
hugs the streets; a convex hull would paint over rivers and campuses nobody can cross (bonus B3 compares hulls).

In [ ]:
def isochrone(graph, node, dist_m, edge_buffer_m=30):
    sub = nx.ego_graph(graph, node, radius=dist_m, distance="length")
    if sub.number_of_edges() == 0:
        x, y = graph.nodes[node]["x"], graph.nodes[node]["y"]
        return Point(x, y).buffer(edge_buffer_m)
    lines = []
    for u, v, data in sub.edges(data=True):
        if "geometry" in data:
            lines.append(data["geometry"])
        else:
            lines.append(shapely.LineString([(graph.nodes[u]["x"], graph.nodes[u]["y"]), (graph.nodes[v]["x"], graph.nodes[v]["y"])]))
    return unary_union([l.buffer(edge_buffer_m) for l in lines])

t0 = time.time()
services_u["iso_15"] = [isochrone(Gu, n, THRESH_M) for n in services_u.node]
services_u["iso_slow"] = [isochrone(Gu, n, THRESH_SLOW_M) for n in services_u.node]
services_u["buf_15"] = services_u.geometry.buffer(THRESH_M)
print(f"{len(services_u)} isochrones x 2 speeds in {time.time()-t0:.0f} s")

iso_union = unary_union(list(services_u.iso_15))
iso_slow_union = unary_union(list(services_u.iso_slow))
buf_union = unary_union(list(services_u.buf_15))
print("share of the ward AREA covered: network %.1f %%  |  buffer %.1f %%  |  network @3.6 km/h %.1f %%" % (
    100 * iso_union.intersection(ward).area / ward.area,
    100 * buf_union.intersection(ward).area / ward.area,
    100 * iso_slow_union.intersection(ward).area / ward.area))

## 3. H3 cells at resolution 8 and 9, with network distance from each cell centre

For every cell centre: snap to the nearest street node, then read the distance to the nearest service from **one**
multi-source Dijkstra run (all service nodes as sources), and add the snap distance. Straight-line distance to the
nearest service is computed too, for the buffer comparison.

In [ ]:
# one Dijkstra from all services at once: distance from every node to its nearest service
dist_to_service = nx.multi_source_dijkstra_path_length(Gu, sources=set(services_u.node), weight="length")

# which service is nearest (needed for Q6): run per-service labelling via a second pass
nearest_service_of_node = {}
for sid, n in zip(services_u.service_id, services_u.node):
    nearest_service_of_node.setdefault(n, sid)

def h3_cells(geom_4326, res):
    shape_ = h3.geo_to_h3shape(geom_4326.__geo_interface__)
    cells = h3.h3shape_to_cells(shape_, res)
    polys = [Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(c)]) for c in cells]
    g = gpd.GeoDataFrame({"h3": list(cells)}, geometry=polys, crs=4326)
    g["res"] = res
    return g

ward_4326 = area.geometry.iloc[0]
hex8 = h3_cells(ward_4326, 8)
hex9 = h3_cells(ward_4326, 9)
print(f"H3 cells whose centre is in the ward: res 8 = {len(hex8)}, res 9 = {len(hex9)}")

svc_tree = shapely.STRtree(services_u.geometry.values)

def measure(hexes):
    hu = hexes.to_crs(UTM)
    cx, cy = hu.geometry.centroid.x.values, hu.geometry.centroid.y.values
    nodes, snap = ox.distance.nearest_nodes(Gu, cx, cy, return_dist=True)
    net = []
    for n, s in zip(nodes, snap):
        d = dist_to_service.get(n, np.inf)
        net.append(np.inf if s > SNAP_MAX_M else d + s)
    hu["node"] = nodes
    hu["snap_m"] = np.round(snap, 1)
    hu["net_dist_m"] = np.round(net, 1)
    nearest_idx = svc_tree.nearest(hu.geometry.centroid.values)
    hu["sl_dist_m"] = np.round(hu.geometry.centroid.distance(services_u.geometry.values[nearest_idx]), 1)
    hu["covered_net_15"] = hu.net_dist_m <= THRESH_M
    hu["covered_buf_15"] = hu.sl_dist_m <= THRESH_M
    hu["covered_net_slow"] = hu.net_dist_m <= THRESH_SLOW_M
    return hu

hex8_u = measure(hex8)
hex9_u = measure(hex9)
hex9_u[["h3", "snap_m", "net_dist_m", "sl_dist_m", "covered_net_15", "covered_buf_15"]].describe(include="all").T.head(8)

## 4. Population per cell (WorldPop 2020, constrained, 100 m)

Each WorldPop pixel is turned into a point at its centre carrying its people count, and the points are summed per
cell. Every person is therefore counted exactly once, whichever cell size is used (the totals at res 8 and res 9
should match).

In [ ]:
wp_path = ROOT / "data" / "worldpop" / "worldpop_prabhag11.tif"
with rasterio.open(wp_path) as src:
    band = src.read(1).astype("float64")
    nodata = src.nodata
    rows, cols = np.where(~np.isnan(band) & (band != nodata) & (band > 0))
    xs, ys = rasterio.transform.xy(src.transform, rows, cols)
pop_pts = gpd.GeoDataFrame({"pop": band[rows, cols]}, geometry=gpd.points_from_xy(xs, ys), crs=4326).to_crs(UTM)
pop_pts_ward = pop_pts[pop_pts.within(ward)].copy()
print(f"WorldPop pixels with people in the window: {len(pop_pts):,}; inside the ward: {len(pop_pts_ward):,} "
      f"= {pop_pts_ward['pop'].sum():,.0f} people")

# pixel-level access (the finest unit we have): distance from each populated pixel centre
px, py = pop_pts_ward.geometry.x.values, pop_pts_ward.geometry.y.values
pnodes, psnap = ox.distance.nearest_nodes(Gu, px, py, return_dist=True)
pop_pts_ward["net_dist_m"] = [np.inf if s > SNAP_MAX_M else dist_to_service.get(n, np.inf) + s for n, s in zip(pnodes, psnap)]
pop_pts_ward["sl_dist_m"] = pop_pts_ward.geometry.distance(services_u.geometry.values[svc_tree.nearest(pop_pts_ward.geometry.values)])

def add_pop(hu):
    j = gpd.sjoin(pop_pts_ward[["pop", "geometry"]], hu[["h3", "geometry"]], predicate="within")
    s = j.groupby("h3")["pop"].sum()
    hu["pop"] = hu["h3"].map(s).fillna(0).round(1)
    return hu
hex8_u = add_pop(hex8_u)
hex9_u = add_pop(hex9_u)
print("population summed over cells: res 8 = %.0f, res 9 = %.0f" % (hex8_u["pop"].sum(), hex9_u["pop"].sum()))

## 5. Q7 — Coverage and the MAUP

"Beyond the threshold" is judged in three ways, from fine to coarse:

* **pixel level** (100 m): each populated pixel by its own network / straight-line distance (reference figure);
* **H3 res 9**, then **res 8**: a cell is covered if its *centre* is within the threshold, and all its people inherit that verdict;
* **ward level**: the whole ward is one unit, covered if its centroid is (the way a ward-level dashboard would show it).

In [ ]:
total = pop_pts_ward["pop"].sum()
def share_beyond(mask_series, pop_series):
    return 100 * pop_series[mask_series].sum() / pop_series.sum()

rows = []
rows.append(("pixel 100 m (reference)", share_beyond(pop_pts_ward.net_dist_m > THRESH_M, pop_pts_ward["pop"]),
             share_beyond(pop_pts_ward.sl_dist_m > THRESH_M, pop_pts_ward["pop"]), len(pop_pts_ward)))
for name, hu in (("H3 res 9", hex9_u), ("H3 res 8", hex8_u)):
    rows.append((name, share_beyond(~hu.covered_net_15, hu["pop"]), share_beyond(~hu.covered_buf_15, hu["pop"]), len(hu)))
# ward as one unit: its centroid
wc = ward.centroid
wn, ws = ox.distance.nearest_nodes(Gu, wc.x, wc.y, return_dist=True)
ward_net = dist_to_service.get(wn, np.inf) + ws
rows.append(("ward (1 unit, centroid)", 100.0 if ward_net > THRESH_M else 0.0,
             100.0 if wc.distance(services_u.geometry.values[svc_tree.nearest(wc)]) > THRESH_M else 0.0, 1))
q7 = pd.DataFrame(rows, columns=["unit", "% population beyond 1200 m (network)", "% population beyond 1200 m (buffer)", "units"])
display(q7.round(1))

worst = hex9_u[hex9_u["pop"] >= 50].sort_values("net_dist_m", ascending=False).head(5)
print("worst-served populated res-9 cells (>= 50 people):")
display(worst[["h3", "pop", "net_dist_m", "sl_dist_m", "snap_m"]])
q7_json = {"total_pop": float(total), "table": q7.round(2).to_dict(orient="records"),
           "worst_cell_r9": worst.iloc[0]["h3"], "worst_cell_r9_pop": float(worst.iloc[0]["pop"]), "worst_cell_r9_net_m": float(worst.iloc[0]["net_dist_m"])}

## 6. Q8 — Who is the model for? Re-run at 3.6 km/h (900 m in 15 minutes)

In [ ]:
rows = []
rows.append(("pixel 100 m", 100 - share_beyond(pop_pts_ward.net_dist_m > THRESH_M, pop_pts_ward["pop"]),
             100 - share_beyond(pop_pts_ward.net_dist_m > THRESH_SLOW_M, pop_pts_ward["pop"])))
for name, hu in (("H3 res 9", hex9_u), ("H3 res 8", hex8_u)):
    rows.append((name, 100 - share_beyond(~hu.covered_net_15, hu["pop"]), 100 - share_beyond(~hu.covered_net_slow, hu["pop"])))
q8 = pd.DataFrame(rows, columns=["unit", "% covered @4.8 km/h", "% covered @3.6 km/h"])
q8["percentage points lost"] = q8.iloc[:, 1] - q8.iloc[:, 2]
display(q8.round(1))

lost = hex9_u[hex9_u.covered_net_15 & ~hex9_u.covered_net_slow]
print(f"res-9 cells that drop out at 3.6 km/h: {len(lost)}, holding {lost['pop'].sum():,.0f} people")
q8_json = {"table": q8.round(2).to_dict(orient="records"), "cells_lost_r9": int(len(lost)), "pop_lost_r9": float(lost["pop"].sum())}

## 7. Q6 — Straight line vs street network for 30 random points

For each random point inside the ward: nearest service by straight line (what PuneStay did) and by walking
network (Dijkstra from the point's nearest node to every service node). Count how often the *identity* of the
nearest service changes, and the ratio network / straight-line.

In [ ]:
rng = random.Random(RNG_SEED)
minx, miny, maxx, maxy = ward.bounds
pts = []
while len(pts) < 30:
    p = Point(rng.uniform(minx, maxx), rng.uniform(miny, maxy))
    if ward.contains(p):
        pts.append(p)
q6 = gpd.GeoDataFrame({"pt_id": range(1, 31)}, geometry=pts, crs=UTM)
qn, qs = ox.distance.nearest_nodes(Gu, q6.geometry.x.values, q6.geometry.y.values, return_dist=True)
q6["node"], q6["snap_m"] = qn, np.round(qs, 1)

svc_by_node = services_u.groupby("node")["service_id"].first().to_dict()
recs = []
for i, row in q6.iterrows():
    # straight line
    d_sl = services_u.geometry.distance(row.geometry)
    j = d_sl.idxmin()
    sl_id, sl_m = int(services_u.loc[j, "service_id"]), float(d_sl[j])
    # network
    lengths = nx.single_source_dijkstra_path_length(Gu, row["node"], weight="length")
    best = None
    for n, sid in svc_by_node.items():
        if n in lengths:
            d = lengths[n] + row["snap_m"] + float(services_u.loc[services_u.service_id == sid, "snap_m"].iloc[0])
            if best is None or d < best[1]:
                best = (sid, d)
    net_id, net_m = (best if best else (None, np.inf))
    # network distance to the straight-line winner, for the ratio
    sl_node = int(services_u.loc[j, "node"])
    net_to_sl_winner = lengths.get(sl_node, np.inf) + row["snap_m"] + float(services_u.loc[j, "snap_m"])
    recs.append(dict(pt_id=row.pt_id, sl_service_id=sl_id, sl_m=round(sl_m, 1), net_service_id=net_id, net_m=round(net_m, 1),
                     net_to_sl_winner_m=round(net_to_sl_winner, 1), ratio=round(net_m / sl_m, 2) if sl_m > 0 else None,
                     different=(sl_id != net_id)))
q6 = q6.join(pd.DataFrame(recs).set_index("pt_id"), on="pt_id")
print(f"nearest service DIFFERENT by network vs straight line: {int(q6.different.sum())} of 30")
print("ratio network / straight-line: median %.2f, max %.2f" % (q6.ratio.median(), q6.ratio.max()))
worst_pt = q6.sort_values("ratio", ascending=False).iloc[0]
print("worst case: point", int(worst_pt.pt_id), "straight %.0f m to service %d, network %.0f m to service %s" % (
    worst_pt.sl_m, worst_pt.sl_service_id, worst_pt.net_m, worst_pt.net_service_id))
display(q6.drop(columns=["geometry", "node"]).sort_values("ratio", ascending=False).head(10))
q6_json = {"different": int(q6.different.sum()), "ratio_median": float(q6.ratio.median()), "ratio_max": float(q6.ratio.max()),
           "worst_pt": int(worst_pt.pt_id), "worst_sl_m": float(worst_pt.sl_m), "worst_net_m": float(worst_pt.net_m),
           "worst_sl_service": int(worst_pt.sl_service_id), "worst_net_service": int(worst_pt.net_service_id) if worst_pt.net_service_id else None}

## 8. Maps

In [ ]:
(ROOT / "docs" / "screenshots").mkdir(parents=True, exist_ok=True)
edges_u = ox.graph_to_gdfs(Gu, nodes=False)

fig, ax = plt.subplots(1, 2, figsize=(15, 7))
for a, (title, col) in zip(ax, [("H3 res 8: network distance to nearest PG/hostel (m)", hex8_u), ("H3 res 9", hex9_u)]):
    col.plot(column=col.net_dist_m.replace(np.inf, np.nan), cmap="RdYlGn_r", vmin=0, vmax=2400, legend=True, ax=a, edgecolor="white", linewidth=0.3)
    gpd.GeoSeries([ward], crs=UTM).boundary.plot(ax=a, color="black", linewidth=1)
    services_u[services_u.inside_area].plot(ax=a, color="blue", markersize=12)
    a.set_title(title); a.set_axis_off()
fig.savefig(ROOT / "docs" / "screenshots" / "q7_h3_network_distance.png", dpi=120, bbox_inches="tight"); plt.show()

fig, ax = plt.subplots(figsize=(9, 9))
gpd.GeoSeries([buf_union.intersection(ward)], crs=UTM).plot(ax=ax, color="#fdd0a2", alpha=0.8, label="buffer 1200 m")
gpd.GeoSeries([iso_union.intersection(ward)], crs=UTM).plot(ax=ax, color="#9ecae1", alpha=0.9, label="isochrone 1200 m")
gpd.GeoSeries([iso_slow_union.intersection(ward)], crs=UTM).plot(ax=ax, color="#3182bd", alpha=0.9, label="isochrone 900 m (3.6 km/h)")
edges_u.clip(ward).plot(ax=ax, color="grey", linewidth=0.3)
gpd.GeoSeries([ward], crs=UTM).boundary.plot(ax=ax, color="black")
services_u[services_u.inside_area].plot(ax=ax, color="red", markersize=18)
ax.set_title("Reachable on foot in 15 min: straight-line buffer (orange) vs street network (blue)"); ax.set_axis_off()
fig.savefig(ROOT / "docs" / "screenshots" / "q7_isochrone_vs_buffer.png", dpi=120, bbox_inches="tight"); plt.show()

# Q6 worst case
fig, ax = plt.subplots(figsize=(8, 8))
wp = q6[q6.pt_id == worst_pt.pt_id]
pad = max(worst_pt.net_m, 600)
view = box(wp.geometry.x.iloc[0] - pad, wp.geometry.y.iloc[0] - pad, wp.geometry.x.iloc[0] + pad, wp.geometry.y.iloc[0] + pad)
edges_u.clip(view).plot(ax=ax, color="grey", linewidth=0.5)
path = nx.shortest_path(Gu, int(wp.node.iloc[0]), int(services_u.loc[services_u.service_id == worst_pt.net_service_id, "node"].iloc[0]), weight="length")
route = shapely.LineString([(Gu.nodes[n]["x"], Gu.nodes[n]["y"]) for n in path])
gpd.GeoSeries([route], crs=UTM).plot(ax=ax, color="blue", linewidth=2.5, label="walking route")
sl_target = services_u[services_u.service_id == worst_pt.sl_service_id].geometry.iloc[0]
gpd.GeoSeries([shapely.LineString([wp.geometry.iloc[0], sl_target])], crs=UTM).plot(ax=ax, color="orange", linewidth=2, linestyle="--", label="straight line")
services_u.clip(view).plot(ax=ax, color="red", markersize=25)
wp.plot(ax=ax, color="black", markersize=40, marker="*")
ax.set_title(f"Q6 worst case: point {int(worst_pt.pt_id)} - straight {worst_pt.sl_m:.0f} m, network {worst_pt.net_m:.0f} m (x{worst_pt.ratio})")
ax.legend(); ax.set_axis_off()
fig.savefig(ROOT / "docs" / "screenshots" / "q6_worst_case.png", dpi=120, bbox_inches="tight"); plt.show()

## 9. Write everything back to PostGIS (EPSG:4326 for the services, UTM kept in a second column)

In [ ]:
def to_db(gdf_u, name, drop=()):
    g = gdf_u.drop(columns=list(drop), errors="ignore").copy()
    g["geom_utm"] = g.geometry
    g = g.set_geometry("geometry").to_crs(4326)
    g = g.replace({np.inf: None})
    with engine.begin() as con:
        g.drop(columns=["geom_utm"]).to_postgis(name, con, if_exists="replace", index=False)
        con.execute(text(f"ALTER TABLE {name} RENAME COLUMN geometry TO geom"))
        con.execute(text(f"ALTER TABLE {name} ADD COLUMN geom_utm geometry(Geometry, {UTM})"))
        con.execute(text(f"UPDATE {name} SET geom_utm = ST_Transform(geom, {UTM})"))
        con.execute(text(f"CREATE INDEX IF NOT EXISTS {name}_geom_gist ON {name} USING GIST (geom)"))
    print("wrote", name, len(g))

to_db(hex8_u, "hex_r8", drop=["node"])
to_db(hex9_u, "hex_r9", drop=["node"])
iso = services_u[["service_id", "name", "service_type", "inside_area"]].copy()
iso = gpd.GeoDataFrame(iso, geometry=services_u["iso_15"].values, crs=UTM); to_db(iso, "isochrones_15min")
iso_s = gpd.GeoDataFrame(services_u[["service_id", "name"]].copy(), geometry=services_u["iso_slow"].values, crs=UTM); to_db(iso_s, "isochrones_15min_slow")
buf = gpd.GeoDataFrame(services_u[["service_id", "name"]].copy(), geometry=services_u["buf_15"].values, crs=UTM); to_db(buf, "buffers_1200m")
to_db(q6, "q6_random_points", drop=["node"])
to_db(pop_pts_ward.replace({np.inf: None}), "worldpop_pixels")

with engine.begin() as con:
    for t in ("hex_r8", "hex_r9"):
        con.execute(text(f"ALTER TABLE {t} ADD PRIMARY KEY (h3)"))
    con.execute(text("ALTER TABLE q6_random_points ADD PRIMARY KEY (pt_id)"))
    con.execute(text("ALTER TABLE isochrones_15min ADD PRIMARY KEY (service_id)"))
    con.execute(text("ALTER TABLE buffers_1200m ADD PRIMARY KEY (service_id)"))

results = {"q6": q6_json, "q7": q7_json, "q8": q8_json, "threshold_m": THRESH_M, "slow_threshold_m": THRESH_SLOW_M,
           "services_total": int(len(services_u)), "services_inside": int(services_u.inside_area.sum()),
           "hex8": int(len(hex8_u)), "hex9": int(len(hex9_u)), "pop_ward": float(total)}
(ROOT / "docs" / "outputs").mkdir(parents=True, exist_ok=True)
(ROOT / "docs" / "outputs" / "phase3_results.json").write_text(json.dumps(results, indent=2), encoding="utf-8")
print(json.dumps(results, indent=2))